In [ ]:
import numpy as np
import pandas as pd


## Observe data

In [ ]:
df = pd.read_csv("/home/tii/datasets/customer_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv", dtype=dict())
print("Desription numbers:\n", df.describe(include="number"))
print()
print("Desription objects:\n", df.describe(include="object"))
print()
df.info()
df.head()

Desription numbers:
        SeniorCitizen       tenure  MonthlyCharges
count    7043.000000  7043.000000     7043.000000
mean        0.162147    32.371149       64.761692
std         0.368612    24.559481       30.090047
min         0.000000     0.000000       18.250000
25%         0.000000     9.000000       35.500000
50%         0.000000    29.000000       70.350000
75%         0.000000    55.000000       89.850000
max         1.000000    72.000000      118.750000

Desription objects:
         customerID gender Partner Dependents PhoneService MultipleLines  \
count         7043   7043    7043       7043         7043          7043   
unique        7043      2       2          2            2             3   
top     7590-VHVEG   Male      No         No          Yes            No   
freq             1   3555    3641       4933         6361          3390   

       InternetService OnlineSecurity OnlineBackup DeviceProtection  \
count             7043           7043         7043          

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.loc[df["TotalCharges"] == " ", "TotalCharges"] = "NaN"

In [4]:
df["TotalCharges"].astype("Float32")

0             29.85
1            1889.5
2        108.150002
3           1840.75
4        151.649994
           ...     
7038         1990.5
7039    7362.899902
7040     346.450012
7041     306.600006
7042         6844.5
Name: TotalCharges, Length: 7043, dtype: Float32

## Generate timestamps

In [5]:
N = 7043
start_date = pd.to_datetime("2026-01-20")
end_date = pd.to_datetime("2026-01-30")

In [6]:
def random_dates(start, end, n):
    start_u = start.value // 10**9
    end_u = end.value // 10**9

    return pd.to_datetime((10**9 * np.random.randint(start_u, end_u, n, dtype=np.int64)).view("M8[ns]"))


In [ ]:
timestamps = random_dates(start=start_date, end=end_date, n=N + 200)
timestamps = timestamps.drop_duplicates()[:N]
timestamps = timestamps.sort_values(ascending=False)
print("size:", timestamps.shape[0], "dtype:", timestamps.dtype)


size: 7043 dtype: datetime64[ns]


## Set timestamps and save results

In [8]:
df["timestamp"] = timestamps.to_pydatetime()
df.loc[:, ["customerID", "timestamp"]].head()

,customerID,timestamp
0,7590-VHVEG,2026-01-29 23:59:40
1,5575-GNVDE,2026-01-29 23:59:15
2,3668-QPYBK,2026-01-29 23:53:50
3,7795-CFOCW,2026-01-29 23:51:36
4,9237-HQITU,2026-01-29 23:43:10


In [9]:
df.dtypes

customerID                  object
gender                      object
SeniorCitizen                int64
Partner                     object
Dependents                  object
tenure                       int64
PhoneService                object
MultipleLines               object
InternetService             object
OnlineSecurity              object
OnlineBackup                object
DeviceProtection            object
TechSupport                 object
StreamingTV                 object
StreamingMovies             object
Contract                    object
PaperlessBilling            object
PaymentMethod               object
MonthlyCharges             float64
TotalCharges                object
Churn                       object
timestamp           datetime64[ns]
dtype: object

In [10]:
df.to_parquet(
    "/home/tii/datasets/customer_churn/telco-customer-churn-with-ts.parquet",
    index=True,
)

## Read result file back

In [11]:
pd.read_parquet("/home/tii/datasets/customer_churn/telco-customer-churn-with-ts.parquet").head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,timestamp
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,2026-01-29 23:59:40
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.5,No,2026-01-29 23:59:15
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,2026-01-29 23:53:50
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,2026-01-29 23:51:36
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,2026-01-29 23:43:10


In [12]:
pd.read_parquet("/home/tii/projects/personal/customer_churn/feature_repo_example/data/driver_stats.parquet").head()

,event_timestamp,driver_id,conv_rate,acc_rate,avg_daily_trips,created
0,2026-01-14 16:00:00+00:00,1005,0.709660,0.372445,618,2026-01-29 16:00:37.023
1,2026-01-14 17:00:00+00:00,1005,0.235640,0.701378,105,2026-01-29 16:00:37.023
2,2026-01-14 18:00:00+00:00,1005,0.215136,0.394129,539,2026-01-29 16:00:37.023
3,2026-01-14 19:00:00+00:00,1005,0.014543,0.687609,302,2026-01-29 16:00:37.023
4,2026-01-14 20:00:00+00:00,1005,0.370674,0.685025,216,2026-01-29 16:00:37.023
